# PyTorch 第十二章：TensorRT 使用

> 对应《PyTorch 实用教程（第二版）》第十二章  
> 目标：建立从 **PyTorch / ONNX → TensorRT Engine → CUDA 推理 → 性能分析 → PTQ / QAT → 工程封装** 的完整部署认知。

## 原教程小节顺序

1. 12.1 TensorRT 简介与安装
2. 12.2 TensorRT 工作流及 cuda-python
3. 12.3 trtexec 工具使用
4. 12.4 TensorRT 实用工具
5. 12.5 TensorRT API 使用
6. 12.6 模型量化基础概念
7. 12.7 PTQ 量化实践
8. 12.8 QAT 量化实践
9. 12.9 TensorRT Python 工程化

教程入口：  
https://tingsongyu.github.io/PyTorch-Tutorial-2nd/chapter-12/

## 本 Notebook 的取舍

TensorRT 强依赖 NVIDIA GPU、CUDA、驱动和 TensorRT 版本，因此：

- 不强制安装 TensorRT，避免 Colab 环境变化导致整本 Notebook 失效；
- TensorRT 专属命令使用可直接复制的模板；
- 核心机制使用 PyTorch / NumPy 构造最小可运行实验；
- 量化部分真正执行 scale / zero-point、calibration、fake quant、PTQ、QAT；
- 对教程 2024 年版本的旧 API 按当前 TensorRT 11.x 思路更新。

### 当前最重要的版本变化

- 原教程主要处于 TensorRT 8.6 / 10.0 生态。
- TensorRT 11.x 已移除旧式 implicit INT8 calibrator API。
- 新项目优先采用 **explicit quantization（Q/DQ）**。
- 当前 TensorRT 支持 FP32 / FP16 / BF16 / FP8 / INT8 / INT4 / FP4 等多种精度路径。
- 推理主路径仍以 tensor name、`set_tensor_address()`、`execute_async_v3()` 为核心。

# 0. 环境导入

In [ ]:
import importlib.util
import math
import random
import shutil
import time
from dataclasses import dataclass

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

HAS_TENSORRT = importlib.util.find_spec("tensorrt") is not None
HAS_CUDA = torch.cuda.is_available()
HAS_TRTEXEC = shutil.which("trtexec") is not None

print("PyTorch:", torch.__version__)
print("CUDA available:", HAS_CUDA)
print("TensorRT Python available:", HAS_TENSORRT)
print("trtexec available:", HAS_TRTEXEC)
print("device:", device)

## 环境策略

TensorRT 不是普通的纯 Python 包：

- pip 安装主要提供 Python bindings / libraries；
- `trtexec` 不保证随 pip wheel 提供；
- 完整 CLI / C++ / 工具链更适合官方容器、Deb/RPM/Tar；
- `.engine/.plan` 是面向 TensorRT Runtime 的序列化引擎，不是像 ONNX 一样的通用交换格式。

因此这一章重点学习**部署机制、性能诊断与量化方法**。

# 12.1 TensorRT 简介与安装

原教程核心：

1. TensorRT 定位
2. Optimizer 与 Runtime
3. 图融合、精度选择、kernel auto-tuning、内存优化
4. Windows / Linux 安装
5. CUDA / TensorRT 版本兼容
6. `trtexec` 验证

## Build 与 Runtime

```text
Build / Optimize
ONNX / Network
   ↓
Graph Optimization
Precision Selection
Kernel / Tactic Selection
Memory Planning
   ↓
Serialized Engine
```

```text
Runtime
Serialized Engine
   ↓
Execution Context
   ↓
CUDA Buffers / Streams
   ↓
Inference
```

构建阶段可以较慢，但通常只做少量次数；Runtime 才是生产高频路径。

In [ ]:
optimization_stack = {
    "graph": ["constant folding", "layer fusion", "dead-node removal"],
    "precision": ["FP16", "BF16", "FP8", "INT8", "INT4/FP4"],
    "kernel": ["tactic selection", "tensor layout"],
    "memory": ["workspace", "buffer reuse"],
    "runtime": ["CUDA stream", "CUDA Graph", "async execution"],
}

for level, items in optimization_stack.items():
    print(f"{level:>10}: {', '.join(items)}")

## 安装建议

真实环境优先顺序通常是：

1. NVIDIA 官方 TensorRT 容器；
2. Linux Deb/RPM/Tar；
3. pip：用于 Python API 实验。

基本验证：

```bash
python -c "import tensorrt as trt; print(trt.__version__)"
trtexec --help
nvidia-smi
```

不要照抄教程中的固定 TensorRT 8.6 / CUDA 11.x 安装组合，版本兼容应以当前官方文档为准。

# 12.2 TensorRT 工作流及 cuda-python

原教程通过 ResNet50 讲解：

- Logger
- Runtime / Builder
- Engine
- ExecutionContext
- Host / Device Buffer
- H2D
- `execute_async_v3`
- D2H
- CUDA Stream
- 资源释放

## Python 推理工作流

```text
serialized engine
   ↓ deserialize
Engine
   ↓ create_execution_context
ExecutionContext
   ↓ set_input_shape
   ↓ set_tensor_address
H2D copy
   ↓
execute_async_v3(stream)
   ↓
D2H copy
   ↓
stream synchronize
```

In [ ]:
@dataclass
class TRTWorkflowState:
    engine_loaded: bool = False
    context_created: bool = False
    buffers_ready: bool = False
    inference_done: bool = False

state = TRTWorkflowState(
    engine_loaded=True,
    context_created=True,
    buffers_ready=True,
    inference_done=True,
)
print(state)

## Host / Device / Stream

- Host Memory：CPU 内存
- Device Memory：GPU 显存
- CUDA Stream：按顺序排队的一组 GPU 工作

异步执行常见逻辑：

```text
H2D → inference → D2H
```

它们放在同一 CUDA stream 时会按顺序执行；最终需要适当同步才能安全读取结果。

In [ ]:
host_input = np.random.randn(2, 4).astype(np.float32)
host_output = np.empty((2, 3), dtype=np.float32)

print("input bytes:", host_input.nbytes)
print("output bytes:", host_output.nbytes)
print("contiguous:", host_input.flags["C_CONTIGUOUS"])

当前 NVIDIA Python 示例更常见：

```python
from cuda.bindings import runtime as cudart
```

核心动作：

```python
cudaMalloc
cudaMemcpyAsync
context.set_tensor_address(...)
context.execute_async_v3(stream)
cudaStreamSynchronize
cudaFree
```

这里不在 Notebook 中强装 TensorRT/CUDA Python，避免版本耦合导致运行失败。

## Dynamic Shape 与 Optimization Profile

动态输入需要：

- `min_shape`
- `opt_shape`
- `max_shape`

其中 `opt_shape` 是 builder 重点进行 tactic 优化的 shape。

In [ ]:
@dataclass(frozen=True)
class ShapeProfile:
    min_shape: tuple
    opt_shape: tuple
    max_shape: tuple

    def contains(self, shape):
        return all(
            lo <= x <= hi
            for x, lo, hi in zip(
                shape, self.min_shape, self.max_shape
            )
        )

profile = ShapeProfile(
    min_shape=(1, 3, 224, 224),
    opt_shape=(8, 3, 224, 224),
    max_shape=(32, 3, 224, 224),
)

for bs in [1, 8, 16, 64]:
    shape = (bs, 3, 224, 224)
    print(shape, "supported:", profile.contains(shape))

要点：

- `min/max`：允许的 runtime shape 范围；
- `opt`：重点调优点，不是“默认 batch”；
- profile 过宽不代表每个 shape 都同样高效；
- 在线服务应让 opt 接近真实高频请求。

# 12.3 trtexec 工具使用

`trtexec` 三个核心用途：

1. ONNX → TensorRT Engine
2. 查看 layer / engine 信息
3. benchmark / profile

## 常用命令模板

固定 shape：

```bash
trtexec --onnx=model.onnx --saveEngine=model.engine
```

动态 shape：

```bash
trtexec   --onnx=model.onnx   --saveEngine=model.engine   --minShapes=input:1x3x224x224   --optShapes=input:8x3x224x224   --maxShapes=input:32x3x224x224
```

精度/分析相关参数常见：

```bash
--fp16
--bf16
--int8
--useCudaGraph
--noDataTransfers
--dumpLayerInfo
--dumpProfile
--exportProfile=profile.json
```

具体参数请以所安装版本的 `trtexec --help` 为准。

## 性能指标

粗略理解：

$$
Latency \approx H2D + GPUCompute + D2H
$$

还应同时看：

- Throughput
- Enqueue Time
- GPU Compute Time
- Total Host Walltime
- P50 / P90 / P95 / P99

In [ ]:
metrics_ms = {
    "enqueue": 0.15,
    "h2d": 0.30,
    "gpu_compute": 1.20,
    "d2h": 0.10,
}

latency = (
    metrics_ms["h2d"]
    + metrics_ms["gpu_compute"]
    + metrics_ms["d2h"]
)

for k, v in metrics_ms.items():
    print(f"{k:>12}: {v:.2f} ms")

print("latency:", round(latency, 2), "ms")

诊断思路：

- GPU Compute 很短、端到端很长 → 传输 / host 开销可能较大；
- Enqueue 很长 → host 可能喂不满 GPU；
- throughput 远低于理论 GPU compute 倒数 → GPU 可能存在空洞或其他瓶颈。

## Layer Fusion

例如：

```text
Conv → BatchNorm → ReLU
```

可能被折叠 / 融合成更少的执行节点，从而减少：

- 中间 tensor；
- global memory 读写；
- kernel launch；
- layout conversion。

# 12.4 TensorRT 实用工具

原教程介绍：

1. Nsight Systems
2. Polygraphy

## Nsight Systems

用于系统级时间线分析，适合回答：

- CPU 是否在等待？
- GPU 是否有空洞？
- memcpy 是否过多？
- 多个 CUDA stream 是否重叠？
- host overhead 是否过大？

典型命令：

```bash
nsys profile -o report python infer.py
```

生产代码常配合 NVTX 标注 preprocess / inference / postprocess。

In [ ]:
def fake_stage(ms):
    time.sleep(ms / 1000)

timeline = {}

start = time.perf_counter()
fake_stage(2)
timeline["preprocess"] = time.perf_counter() - start

start = time.perf_counter()
fake_stage(3)
timeline["inference"] = time.perf_counter() - start

start = time.perf_counter()
fake_stage(1)
timeline["postprocess"] = time.perf_counter() - start

for name, seconds in timeline.items():
    print(name, f"{seconds * 1000:.2f} ms")

## Polygraphy

适合模型转换与数值调试：

- ONNX Runtime vs TensorRT 输出比较；
- 找到误差从哪一层开始；
- inspect 模型；
- 图裁剪 / graph surgery；
- parser / tactic / precision 问题隔离。

常见子命令：

```bash
polygraphy run
polygraphy convert
polygraphy inspect
polygraphy surgeon
polygraphy debug
polygraphy data
```

核心原则：

> 先确认转换后的结果正确，再讨论速度。

# 12.5 TensorRT API 使用

构建 TensorRT Network 常见三条路径：

1. TensorRT 原生 API
2. ONNX Parser
3. 框架集成工具

ONNX Parser 通常最方便；原生 API 最灵活但维护成本最高。

## 核心对象

```text
Logger
 ↓
Builder
 ↓
NetworkDefinition
 ↓
BuilderConfig
 ↓
OptimizationProfile
 ↓
Serialized Engine
 ↓
Runtime / Engine
 ↓
ExecutionContext
```

原生 API 的基本抽象：

> `ITensor → ILayer → ITensor`

In [ ]:
@dataclass
class TensorDesc:
    name: str
    shape: tuple

@dataclass
class LayerDesc:
    op: str
    input_name: str
    output_name: str

x_desc = TensorDesc("input", (1, 3, 224, 224))
layers = [
    LayerDesc("Conv", "input", "conv_out"),
    LayerDesc("ReLU", "conv_out", "relu_out"),
    LayerDesc("Pool", "relu_out", "pool_out"),
]

print(x_desc)
for layer in layers:
    print(layer)

## Tactic / Kernel Auto-Tuning

同一个算子可能存在多个底层实现。Builder 会综合：

- GPU 架构
- tensor shape
- precision
- workspace
- tensor layout

选择更优 tactic。

因此 Engine 构建结果与目标硬件和构建配置强相关。

In [ ]:
tactics_ms = {
    "tactic_A": 0.42,
    "tactic_B": 0.31,
    "tactic_C": 0.37,
}
best = min(tactics_ms, key=tactics_ms.get)

print("best tactic:", best)
print("latency:", tactics_ms[best], "ms")

## `.plan` 与 `.engine`

实践中它们都常用来表示 serialized TensorRT engine。

不要把 `.plan` 理解成像 ONNX 一样可随意跨硬件使用的“通用模型格式”。

更安全的默认认知：

> TensorRT Engine 与 TensorRT 版本、GPU 架构、构建配置相关。

生产发布最好同时保留 ONNX/source model，必要时在目标环境重新构建 Engine。

# 12.6 模型量化基础概念

原教程重点：

1. 量化目的
2. scale / zero-point
3. 对称 / 非对称
4. dynamic range
5. per-tensor / per-channel
6. PTQ / QAT
7. fake quantization

## Affine Quantization

$$
q=\mathrm{round}\left(\frac{r}{S}+Z\right)
$$

$$
\hat r=S(q-Z)
$$

- $r$：原始浮点值
- $q$：量化整数
- $S$：scale
- $Z$：zero-point
- $\hat r$：反量化后的近似值

In [ ]:
def affine_quantize(x, qmin=-128, qmax=127):
    x = x.float()
    rmin = x.min()
    rmax = x.max()

    scale = (rmax - rmin) / (qmax - qmin)
    if float(scale) == 0.0:
        scale = torch.tensor(1.0)

    zero_point = torch.round(qmin - rmin / scale)
    zero_point = torch.clamp(zero_point, qmin, qmax)

    q = torch.round(x / scale + zero_point)
    q = torch.clamp(q, qmin, qmax)

    x_hat = scale * (q - zero_point)
    return q.to(torch.int8), scale, zero_point, x_hat

x = torch.tensor([-1.2, -0.3, 0.0, 0.8, 2.1])
q, scale, zp, x_hat = affine_quantize(x)

print("x:", x)
print("q:", q)
print("scale:", float(scale))
print("zero-point:", int(zp))
print("x_hat:", x_hat)
print("MAE:", float((x - x_hat).abs().mean()))

## 对称 INT8

当前 TensorRT INT8 explicit quantization 采用对称量化思路。

简单形式：

$$
S=\frac{\max |x|}{127}
$$

此时 zero-point 为 0。

In [ ]:
def symmetric_int8_quantize(x, amax=None):
    x = x.float()

    if amax is None:
        amax = x.abs().max()

    scale = amax / 127.0
    if float(scale) == 0.0:
        scale = torch.tensor(1.0, device=x.device)

    q = torch.round(x / scale).clamp(-128, 127).to(torch.int8)
    x_hat = q.float() * scale
    return q, scale, x_hat

q_sym, s_sym, x_sym = symmetric_int8_quantize(x)

print("q:", q_sym)
print("scale:", float(s_sym))
print("MAE:", float((x - x_sym).abs().mean()))

## 位宽变小 ≠ 一定按比例加速

FP32 → INT8 的理论权重存储从 4 bytes/element 降到约 1 byte/element。

但真实速度取决于：

- GPU 是否有高效低精度 kernel；
- 算子和 shape；
- Tensor Core 使用；
- Q/DQ 开销；
- layer fusion；
- 数据搬运。

因此只能 benchmark，不能把“理论位宽比”当成实际加速倍数。

## Per-Tensor vs Per-Channel

- activation 常见 per-tensor；
- weight 常见 per-channel / per-axis；
- per-channel 更能适配不同输出通道的数值范围。

In [ ]:
weight = torch.tensor([
    [0.01, 0.02, -0.03],
    [1.20, -0.80, 0.50],
])

_, _, w_tensor = symmetric_int8_quantize(weight)

row_amax = weight.abs().amax(dim=1, keepdim=True)
row_scale = row_amax / 127
q_channel = torch.round(weight / row_scale).clamp(-128, 127)
w_channel = q_channel * row_scale

print("per-tensor MAE:",
      float((weight - w_tensor).abs().mean()))
print("per-channel MAE:",
      float((weight - w_channel).abs().mean()))

## Calibration Range

绝对最大值不一定是最佳量化范围。

若绝大多数 activation 在 `[-2,2]`，但少量 outlier 到 `±20`：

- max calibration：无 clipping，但主体量化步长很粗；
- percentile / MSE / entropy：允许少量 clipping，换取主体更细的量化分辨率。

In [ ]:
torch.manual_seed(42)

normal = torch.randn(10000)
outliers = torch.tensor([15.0, -18.0, 20.0])
activation = torch.cat([normal, outliers])

amax_max = activation.abs().max()
amax_p999 = torch.quantile(activation.abs(), 0.999)

_, _, rec_max = symmetric_int8_quantize(
    activation, amax=amax_max
)
_, _, rec_p = symmetric_int8_quantize(
    activation, amax=amax_p999
)

rec_p = rec_p.clamp(-amax_p999, amax_p999)

print("max range:", float(amax_max))
print("99.9% range:", round(float(amax_p999), 3))
print("bulk MAE (max):",
      round(float((normal - rec_max[:len(normal)]).abs().mean()), 5))
print("bulk MAE (percentile):",
      round(float((normal - rec_p[:len(normal)]).abs().mean()), 5))

结论：

> calibration 是 clipping error 与 quantization resolution 的折中。

原教程进一步讨论 max / entropy / MSE / percentile，它们都在解决“如何选择有效动态范围”。

# 12.7 PTQ 量化实践

PTQ = Post-Training Quantization。

原教程流程：

1. 已训练 FP32 模型
2. 替换 / 插入量化模块
3. calibration data 前向统计
4. 计算动态范围
5. 得到 scale
6. 导出 ONNX
7. TensorRT 构建 INT8 engine
8. 对比精度与性能

## 一个最小 PTQ 实验

这里用小 MLP：

- 不重新训练；
- 用 calibration data 收集中间 activation；
- 根据 percentile 得到 activation range；
- forward 中执行 fake INT8。

In [ ]:
torch.manual_seed(42)

class SmallMLP(nn.Module):
    def __init__(self):
        super().__init__()
        self.fc1 = nn.Linear(4, 16)
        self.fc2 = nn.Linear(16, 3)

    def forward(self, x):
        return self.fc2(F.relu(self.fc1(x)))

fp_model = SmallMLP().eval()
calibration_data = torch.randn(256, 4)
test_data = torch.randn(64, 4)

with torch.inference_mode():
    fp_logits = fp_model(test_data)

print("FP output:", fp_logits.shape)

In [ ]:
activation_values = []

def collect_activation(module, inputs, output):
    activation_values.append(output.detach().flatten())

handle = fp_model.fc1.register_forward_hook(collect_activation)

with torch.inference_mode():
    _ = fp_model(calibration_data)

handle.remove()

activations = torch.cat(activation_values)

print("num activation values:", activations.numel())
print("amax:", round(float(activations.abs().max()), 4))
print("p99.9:", round(
    float(torch.quantile(activations.abs(), 0.999)), 4
))

In [ ]:
class PTQSimulatedMLP(nn.Module):
    def __init__(self, original, activation_amax):
        super().__init__()
        self.fc1 = original.fc1
        self.fc2 = original.fc2
        self.activation_amax = float(activation_amax)

    def forward(self, x):
        x = self.fc1(x)
        _, _, x = symmetric_int8_quantize(
            x,
            amax=torch.tensor(
                self.activation_amax,
                device=x.device,
            ),
        )
        x = F.relu(x)
        return self.fc2(x)

amax = torch.quantile(activations.abs(), 0.999)
ptq_model = PTQSimulatedMLP(fp_model, amax).eval()

with torch.inference_mode():
    ptq_logits = ptq_model(test_data)

print(
    "mean abs output error:",
    round(float((fp_logits - ptq_logits).abs().mean()), 6),
)

## Calibration Dataset

校准数据可以不需要标签，但必须代表真实部署分布。

需要检查：

- 与生产一致的预处理；
- 足够覆盖输入分布；
- 图像亮度/类别/场景；
- NLP sequence length；
- 动态 shape 范围。

PTQ 掉点严重时，优先检查 calibration data 与 range，而不是直接认为“INT8 不适合”。

## 旧式 TensorRT PTQ 已经发生关键变化

原教程使用的旧路线包含：

```text
IInt8EntropyCalibrator
IInt8MinMaxCalibrator
setInt8Calibrator()
dynamic range API
```

TensorRT 11.x 已移除 implicit quantization / calibrator API。

现代路线应理解为：

```text
PyTorch model
 ↓
Model Optimizer / framework quantization
 ↓
explicit Q/DQ
 ↓
ONNX
 ↓
TensorRT
```

# 12.8 QAT 量化实践

QAT = Quantization-Aware Training。

核心：

> 在训练 forward 中模拟量化误差，让参数主动适应低比特表示。

## Fake Quant 与 STE

Forward：

$$
x \rightarrow Q(x)\rightarrow DQ(Q(x))\approx x
$$

rounding 的真实梯度几乎处处为 0，因此常用 Straight-Through Estimator：

- forward：真的 round / clip；
- backward：近似把梯度直接传回。

In [ ]:
class FakeQuantSTE(torch.autograd.Function):
    @staticmethod
    def forward(ctx, x, scale):
        q = torch.round(x / scale).clamp(-127, 127)
        return q * scale

    @staticmethod
    def backward(ctx, grad_output):
        return grad_output, None

def fake_quant_ste(x, amax):
    scale = torch.clamp(amax / 127.0, min=1e-8)
    return FakeQuantSTE.apply(x, scale)

x_ste = torch.tensor(
    [0.13, 0.37, -0.84],
    requires_grad=True,
)

y_ste = fake_quant_ste(x_ste, torch.tensor(1.0))
y_ste.sum().backward()

print("fake quant:", y_ste.detach())
print("gradient:", x_ste.grad)

## 最小 QAT 训练

目标：

$y = 2x_1 - 3x_2 + 0.5$

训练时每次 forward 都把权重 fake quantize。

In [ ]:
torch.manual_seed(42)

X_train = torch.randn(512, 2)
y_train = (
    2 * X_train[:, :1]
    - 3 * X_train[:, 1:2]
    + 0.5
)

class QATLinear(nn.Module):
    def __init__(self):
        super().__init__()
        self.linear = nn.Linear(2, 1)

    def forward(self, x):
        w = self.linear.weight
        amax = w.detach().abs().max().clamp(min=1e-8)
        w_q = fake_quant_ste(w, amax)
        return F.linear(x, w_q, self.linear.bias)

qat_model = QATLinear()
optimizer = torch.optim.Adam(
    qat_model.parameters(),
    lr=0.03,
)

for step in range(200):
    pred = qat_model(X_train)
    loss = F.mse_loss(pred, y_train)

    optimizer.zero_grad()
    loss.backward()
    optimizer.step()

print("final loss:", round(float(loss), 6))
print("weight:", qat_model.linear.weight.detach())
print("bias:", qat_model.linear.bias.detach())

## Q/DQ 的部署意义

```text
FP tensor
 ↓ Quantize
INT8 / low precision
 ↓ low-precision op
INT8 / low precision
 ↓ Dequantize
FP tensor
```

TensorRT explicit quantization 根据 Q/DQ 节点语义决定精度转换，并在允许情况下融合。

当前 TensorRT 还支持 FP8、INT4、FP4 等路径，但它们的硬件要求和量化规则不同。

## PTQ vs QAT

| 维度 | PTQ | QAT |
|---|---|---|
| 是否重新训练 | 通常不需要 | 需要 |
| 额外数据 | calibration data | 训练数据 |
| 工程成本 | 较低 | 较高 |
| 精度恢复能力 | 中等 | 通常更好 |
| 使用策略 | 先尝试 | PTQ 不达标再考虑 |

工程上通常优先 PTQ，而不是默认所有模型都 QAT。

# 12.9 TensorRT Python 工程化

原教程最后以 YOLOv5 为例，关注如何把 engine 推理封装成生产模块。

当前设计应优先围绕：

- tensor name API
- `set_tensor_address`
- `execute_async_v3`
- buffer reuse
- CUDA stream
- dynamic shape

## 一个合理的封装职责

```text
TRTEngine
├── __init__
│   ├── deserialize engine
│   ├── create context
│   ├── create CUDA stream
│   ├── inspect I/O tensors
│   └── allocate reusable buffers
├── infer
│   ├── validate input
│   ├── set_input_shape
│   ├── H2D
│   ├── set_tensor_address
│   ├── execute_async_v3
│   ├── D2H
│   └── synchronize
└── close
    └── release resources
```

In [ ]:
class MockTRTEngine:
    # 不依赖 TensorRT，只模拟生产类的职责分离。

    def __init__(self, input_shape, output_shape):
        self.input_shape = input_shape
        self.output_shape = output_shape
        self.calls = 0

    def validate(self, x):
        if tuple(x.shape[1:]) != tuple(self.input_shape[1:]):
            raise ValueError(
                f"expected trailing shape {self.input_shape[1:]}, "
                f"got {tuple(x.shape[1:])}"
            )
        if x.dtype != np.float32:
            raise TypeError("expected float32")

    def infer(self, x):
        self.validate(x)
        self.calls += 1
        batch = x.shape[0]
        return np.zeros(
            (batch, self.output_shape[-1]),
            dtype=np.float32,
        )

engine = MockTRTEngine(
    input_shape=(-1, 4),
    output_shape=(-1, 3),
)

x_mock = np.random.randn(5, 4).astype(np.float32)
y_mock = engine.infer(x_mock)

print("output:", y_mock.shape)
print("calls:", engine.calls)

## Buffer 复用

不要每个请求都：

```text
cudaMalloc → infer → cudaFree
```

更合理：

- 固定 shape：初始化时一次分配；
- 动态 shape：按 profile / bucket 分配；
- 超出容量时再扩容；
- 使用 memory pool。

In [ ]:
class BufferPool:
    def __init__(self):
        self.pool = {}

    def get(self, shape, dtype=np.float32):
        key = (tuple(shape), np.dtype(dtype).str)

        if key not in self.pool:
            self.pool[key] = np.empty(
                shape,
                dtype=dtype,
            )
            created = True
        else:
            created = False

        return self.pool[key], created

pool = BufferPool()

for shape in [(1, 4), (1, 4), (8, 4), (1, 4)]:
    _, created = pool.get(shape)
    print(shape, "new allocation:", created)

## 端到端性能不只等于 Engine 性能

目标检测服务：

```text
decode image
 ↓
resize / normalize
 ↓
H2D
 ↓
TensorRT
 ↓
D2H
 ↓
decode boxes
 ↓
NMS
```

即使 engine 加速 4 倍，如果模型计算只占总耗时的一半，端到端也不可能加速 4 倍。

In [ ]:
def amdahl_speedup(accelerated_fraction, local_speedup):
    return 1 / (
        (1 - accelerated_fraction)
        + accelerated_fraction / local_speedup
    )

for fraction in [0.5, 0.8, 0.95]:
    total = amdahl_speedup(
        accelerated_fraction=fraction,
        local_speedup=4,
    )
    print(
        f"计算占比={fraction:.0%}, "
        f"局部加速4x -> 端到端约 {total:.2f}x"
    )

## 生产边界条件

至少处理：

- TensorRT / CUDA 初始化失败；
- engine 与环境不兼容；
- input name / dtype 错误；
- shape 超出 optimization profile；
- GPU OOM；
- `execute_async_v3()` 失败；
- CUDA stream 同步；
- dynamic output shape；
- 多线程 / 多 context 并发安全；
- warm-up；
- profiling 与日志。

# 当前 TensorRT 与 LLM 的补充

原教程以 ResNet / YOLO 为主要案例。

当前 TensorRT 已进一步支持：

- fused attention；
- transformer-specific optimization；
- KV Cache；
- MoE；
- ragged batching；
- multi-device inference；
- FP8 / FP4 / INT4。

专门部署 LLM 时，还应继续学习：

- TensorRT-LLM
- continuous batching
- paged / managed KV Cache
- speculative decoding
- tensor parallel
- serving scheduler

这些超出原教程第十二章范围。

# 完整部署链路

```text
PyTorch
 ↓
ONNX export
 ↓
ONNX / ORT numerical check
 ↓
Polygraphy debug
 ↓
TensorRT build
 ├─ dynamic profiles
 ├─ precision
 ├─ tactics
 └─ fusion
 ↓
Serialized Engine
 ↓
Runtime Wrapper
 ├─ reusable buffers
 ├─ CUDA stream
 └─ execute_async_v3
 ↓
Benchmark
 ├─ latency
 ├─ throughput
 ├─ H2D / D2H
 └─ percentiles
 ↓
Nsight Systems / profiling
 ↓
single-variable optimization
```

# 量化知识结构

```text
FP model
 ↓
choose calibration range
 ↓
scale / zero-point
 ↓
PTQ or QAT
 ↓
Q/DQ graph
 ↓
ONNX
 ↓
TensorRT explicit quantization
 ↓
low-precision kernels
```

必须区分：

- PTQ：训练结束后量化；
- QAT：训练中模拟量化误差；
- Fake Quant：forward 模拟 Q→DQ，但 tensor 仍以浮点参与训练；
- Explicit Quantization：Q/DQ 明确存在于部署计算图中。

# 学完必须会回答的 12 个问题

1. TensorRT Build 阶段和 Runtime 阶段分别负责什么？
2. 为什么 TensorRT Engine 不能直接当成类似 ONNX 的通用跨平台格式？
3. Builder、Network、BuilderConfig、Engine、ExecutionContext 分别是什么？
4. dynamic shape 中 min / opt / max 分别有什么作用？
5. `trtexec` 最重要的三个用途是什么？
6. Throughput、Latency、GPU Compute、H2D/D2H、Enqueue Time 分别说明什么？
7. Nsight Systems 和 Polygraphy 分别用于什么？
8. 为什么 TensorRT 要做 tactic / kernel auto-tuning？
9. 对称量化、非对称量化、per-tensor、per-channel 分别是什么？
10. calibration range 为什么不一定直接使用绝对最大值？
11. PTQ 与 QAT 的核心差异是什么？Fake Quant / STE 的意义是什么？
12. 为什么生产封装要复用 Engine、Context、Stream、Buffer？

# 面向后续 LLM / AI 工程的复习优先级

1. **PyTorch → ONNX → TensorRT 整体链路**
2. **latency / throughput / benchmark**
3. **dynamic shapes / optimization profile**
4. **FP16 / INT8 / PTQ / QAT / Q-DQ**
5. **CUDA buffer + stream + async inference**
6. **Polygraphy / Nsight 调试思路**
7. **Engine wrapper 工程设计**
8. 原生逐层 TensorRT API：需要时再查

后续若走 LLM 部署，再系统学习 TensorRT-LLM、vLLM、SGLang 会更有价值。